## Урок 8. Retrieval‑Augmented Generation (RAG)

В этом уроке мы сделаем из LLM ассистента, который сможет отвечать на русском на произвольные вопросы о фильмах. Для этого мы будем использовать RAG и датасет с отзывами от Кинопоиска.

### План

1. Как модель работает без RAG?
1. Строим RAG на отзывах
1. Reranker
1. Multi-Query
1. Фильтрация по мета-информации

In [6]:
!pip install -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cu

In [7]:
import numpy as np
import uuid
from tqdm.auto import tqdm

### Загрузка датасета и модели

Скачаем [датасет](https://huggingface.co/datasets/blinoff/kinopoisk) из HugingFace Hub. Он содержит больше 35 тысяч отзывов пользователей на различные фильмы.

In [8]:
from datasets import load_dataset

def process_dataset(sample):
    sample['content'] = sample['content'].replace('\xa0', ' ')
    return sample

dataset = load_dataset("blinoff/kinopoisk")['train']
dataset = dataset.map(process_dataset)

README.md:   0%|          | 0.00/1.31k [00:00<?, ?B/s]

kinopoisk.jsonl:   0%|          | 0.00/143M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/36591 [00:00<?, ? examples/s]

Map:   0%|          | 0/36591 [00:00<?, ? examples/s]

In [4]:
dataset

Dataset({
    features: ['part', 'movie_name', 'review_id', 'author', 'date', 'title', 'grade3', 'grade10', 'content'],
    num_rows: 36591
})

In [9]:
dataset[0]['movie_name']

'Блеф (1976)'

In [10]:
dataset[0]['content']

'\n"Блеф» — одна из моих самых любимых комедий.\n\nЭтот фильм я наверно смотрел раз сто, нет я конечно блефую, я видел его куда больше. Не могу не выразить своё восхищение главными действующими лицами этого фильма. Начну с Адриано Челентано для которого как я считаю это лучшая роль в кино. Великолепный актёр, неплохой певец, странно что на его родине в Италии его песни мало кто слушает. Ну я думаю что и итальянцы и французы привыкли к тому, что у нас до сих их актёры популярней чем даже на своей родине. Да, такой вот парадокс. Челентано конечно профессионал своего дела, комик с серьёзным выражением лица. Он смешон ещё и потому, что одновременно так серъёзен. Адриано браво!\n\nА теперь несколько слов об Энтони Куине. Да тот самый горбун из Нотр-дама. Собор Парижской Богоматери, оригинальная версия, кто не смотрел рекомендую. С ним как-то приключилась одна интересная история. На съёмках одного из своих фильмов он то ли сломал, то ли подвихнул ногу, а роль требовала от него чтобы в одной 

Мы будем проводить все испытания с моделью [`Qwen/Qwen2-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2-1.5B-Instruct). Это вопросно-ответная модель на основе GPT, которая обучалась на большом количестве языков, в том числе на русском.

In [11]:
from transformers import pipeline
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

generation_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2-1.5B-Instruct",
    device=device,
    torch_dtype=torch.float16
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Device set to use cuda


## Генерация без RAG

Проверим, насколько хорошо модель отвечает на вопросы без использования RAG.

In [12]:
query = 'В каких пяти фильмах играл Роберт де Ниро?'

In [13]:
messages = [
    {"role": "user", "content": query},
]
output = generation_pipeline(messages, max_new_tokens=256, do_sample=True, temperature=0.9, top_p=0.7)

answer = output[0]['generated_text'][1]['content']

print(answer)

Роберт Де Ниро вел несколько крупных ролей в кино, но он был актером главного роля в следующих пяти фильмах:

1. "Люди в черном" (The Godfather) - 1972 год.
2. "Игра престолов" (Game of Thrones) - 2013-2019 годы.
3. "Джон Гуд" (There Will Be Blood) - 2007 год.
4. "Ночной дьявол" (The Departed) - 2006 год.
5. "Ангелы и демони" (The Aviator) - 2004 год.

Пожалуйста, убедитесь, что вы знаете все возможные ответы. Я не могу знать прошлое или будущее.


Видим, что знаний модели не хватает. Все названия, кроме Терминатора, отсылают к несуществующим фильмам, а в Терминаторе Роберт де Ниро не играл.

##  Retrieval‑Augmented Generation

Попробуем улучшить качество модели с помощью RAG. Для этого нам сперва надо составить векторную базу данных. В качестве эмбеддинговой модели будем использовать [`intfloat/multilingual-e5-large`](https://huggingface.co/intfloat/multilingual-e5-large). Это большая мультиязычная модель на основе Encoder'a Трансформера.

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/multilingual-e5-large", model_kwargs={'torch_dtype': torch.float16})

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Мы построим нашу базу данных на основе инструмента [`Qdrant`](https://github.com/qdrant/qdrant-client). В нем реализованы различные методы для поиска текстов, так что не придется писать ничего руками. Достаточно передать векторы эмбеддингов.

In [15]:
!pip install -q qdrant_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.7/327.7 kB 21.3 MB/s eta 0:00:00


In [16]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="kinopoisk_e5",
    on_disk_payload=True,
    vectors_config=models.VectorParams(
        size=1024,
        distance=models.Distance.COSINE,
        on_disk=True
    ),
)

True

Для разбиения текста на куски используем `RecursiveCharacterTextSplitter` из библиотеки [`langchain`](https://github.com/langchain-ai/langchain). Мы будем делить текст на куски примерно по 1000 символов рекурсивно.

In [17]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

Собираем векторную базу данных.

In [18]:
for i in tqdm(range(len(dataset))):
    # разбиваем текст на куски
    text_chunks = text_splitter.split_text(dataset[i]['content'])

    # кодируем все куски в эмбеддинги
    vectors = embedding_model.encode(text_chunks, normalize_embeddings=True, device=device).tolist()

    # добавляем результат в базу данных
    client.upsert(
        collection_name='kinopoisk_e5',
        points=[
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=vectors[j],
                payload={
                    # дополнительно сохраняем сам текст, название фильма и год выпуска
                    'text': text_chunks[j],
                    'movie_name': dataset[i]['movie_name'][:-7],
                    'year': int(dataset[i]['movie_name'][-5:-1]),
                }
            )
            for j in range(len(text_chunks))
        ]
    )

  0%|          | 0/36591 [00:00<?, ?it/s]

<ipython-input-18-9fef0f7ca02a>:9: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 20001 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Проверим, насколько хорошо ищутся похожие по смыслу тексты.

In [19]:
query_vector = embedding_model.encode(query, normalize_embeddings=True, device=device).tolist()

In [20]:
hits = client.search(
    collection_name="kinopoisk_e5",
    query_vector=query_vector,
    limit=5
)

<ipython-input-20-17d4700f61af>:1: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


In [21]:
[hit.payload for hit in hits]

[{'text': 'Роберт де Ниро сыграл потрясающую роль, передал всю боль, все счастье, все отчаяние больного.\n\nЭтот фильм должен посмотреть каждый. Возможно, кто-то сможет «пробудиться» к настоящей жизни.\n\n9 из 10',
  'movie_name': 'Пробуждение',
  'year': 1990},
 {'text': 'Потрясающий фильм. Великолепная игра Роберта Де Ниро. Изумительная по красоте и, как нельзя подходящая к этой кинокартине, музыка Эннио Морриконе. Этот шедевр стоит того, чтобы смотреть и смотреть не один раз.',
  'movie_name': 'Однажды в Америке',
  'year': 1983},
 {'text': 'Хороший фильм. На реальных событиях. Игра де Ниро восхищает. Робин Уильямс как обычно смешной и трагический.',
  'movie_name': 'Пробуждение',
  'year': 1990},
 {'text': 'После долгих и продуктивных лет работы в качестве актера, Роберт де Ниро решил попробовать себя и в режиссерском поприще. Первый фильм этого гениального человека собрал в Америке больше 17 миллионов долларов. Интригующее повествование о судьбе обычного молодого парня, которому п

Ура! Все полученные тексты связаны с Робертом де Ниро. Теперь обернем этот процесс поиска в функцию.

In [22]:
def semantic_search(client, query, limit=10):
    query_vector = embedding_model.encode(
        query, normalize_embeddings=True, device=device
    ).tolist()

    hits = client.search(
        collection_name="kinopoisk_e5",
        query_vector=query_vector,
        limit=limit
    )
    relevant_chunks = [hit.payload for hit in hits]

    return relevant_chunks

### RAG на отзывах

Для реализации RAG будем передавать в контекст модели отзывы, относящиеся к запросу.

In [23]:
def llm_answer(query, context):
    prompt = f"""
    Ты русскоязычный эксперт в области кинематографа. У тебя есть доступ к набору отзывов о фильмах, используй их, чтобы полно и точно ответить на следующий вопрос. Убедись, что ответ подробный, конкретный и непосредственно касается вопроса. Не добавляй информацию, которая не подтверждается предоставленными отзывами.

Вопрос:
{query}

Отзывы:
{context}
"""
    messages = [
        {"role": "user", "content": prompt},
    ]
    output = generation_pipeline(messages, max_new_tokens=512, do_sample=True, temperature=0.9, top_p=0.7)

    return output[0]['generated_text'][1]['content']

In [24]:
def predict(query):
    selected_chunks = semantic_search(client, query)
    context = ' ; '.join([f"Отзыв: {chunk['text']}" for chunk in selected_chunks])

    return llm_answer(query, context)

In [25]:
print(predict(query))

<ipython-input-22-a1fa2b3a3849>:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


Роберт де Ниро играл в следующих пяти фильмах:

1. "Терминатор 2: Геноцид" (Terminator 2: Judgment Day)
2. "Абакос" (Abacá)
3. "Лучшие друзья" (The Best of the Worst)
4. "Оскар" (Oscar)
5. "Золотой медведь" (Golden Eagle)


Несмотря не подсказки, получилось так себе. Дело в том, что отзывы очень редко содержат сами названия фильмов. Попробуем добавить их тоже контекст.

### Добавляем названия фильмов

In [26]:
def predict(query):
    selected_chunks = semantic_search(client, query)
    context = ' ; '.join([f"Название: {chunk['movie_name']}. Отзыв: {chunk['text']}" for chunk in selected_chunks])

    return llm_answer(query, context)

In [27]:
print(predict(query))

<ipython-input-22-a1fa2b3a3849>:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


Роберт де Ниро играл в следующих пяти фильмах:

1. Пробуждение (2001)
2. Однажды в Америке (1999)
3. Пробуждение (2004)
4. Бронкская история (2006)
5. Схватка (2012)


Теперь модель возвращает все фильмы, в которых Роберт де Ниро действительно играл.

## Reranker

Давайте попробуем улучшить метод, добавив переранжирование отзызов.

<img src="https://i.ibb.co/txhQby0/reranker.png" alt="drawing" width="700"/>


Reranker – это языковая модель, принимающая два текста и возвращающая близость между ними.

<img src="https://i.ibb.co/RcPGVs5/reranker-model.png" alt="drawing" width="200"/>

В качестве реранкера мы будем использовать специальную мультиязычную модель, обученную для этих целей [`amberoad/bert-multilingual-passage-reranking-msmarco`](https://huggingface.co/amberoad/bert-multilingual-passage-reranking-msmarco).

In [4]:
!pip install langchain_community langchain pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 99.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.6 MB/s eta 0:00:00


In [7]:
!pip uninstall -y pydantic

Found existing installation: pydantic 2.11.4
Uninstalling pydantic-2.11.4:
  Successfully uninstalled pydantic-2.11.4


In [2]:
import langchain
import langchain_core
import pydantic

print(f"langchain: {langchain.__version__}")
print(f"langchain_core: {langchain_core.__version__}")
print(f"pydantic: {pydantic.__version__}")

langchain: 0.3.24
langchain_core: 0.3.56
pydantic: 2.11.4


In [28]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

In [29]:
cross_encoder = HuggingFaceCrossEncoder(
    model_name='amberoad/bert-multilingual-passage-reranking-msmarco',
    model_kwargs={'device': 'cpu'}
)
sum([p.numel() for p in cross_encoder.client.model.parameters()])

config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/669M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

167357954

In [30]:
def predict(query):
    selected_chunks = semantic_search(client, query, limit=50)

    texts = [f"Название: {chunk['movie_name']}. Отзыв: {chunk['text']}" for chunk in selected_chunks]
    scores = cross_encoder.score([(query, text) for text in texts])

    idxs = np.argsort(list(scores))[-10:]

    context = ' ; '.join([texts[i] for i in idxs])
    return llm_answer(query, context), context

In [31]:
answer, context = predict(query)

print(answer)

<ipython-input-22-a1fa2b3a3849>:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Роберт Де Ниро играл в следующих пяти фильмах:

1. Бронкская история (The Godfather)
2. Однажды в Америке (Once Upon a Time in America)
3. Пробуждение (The Aviator)
4. Военный ныряльщик (Patton)
5. Пробуждение (In the Name of the Father)

Также стоит отметить, что он также играл в нескольких других фильмах, таких как "Крестный отец" (Crazy Heart), "Славные парни" (The Good Guys), "Бронкская история" и другие.


Стало действительно немного лучше.

In [32]:
context.split(' ; ')

['Название: Бронкская история. Отзыв: Богатейший актерский опыт Роберта Де Ниро с такими авторитетными режиссерами криминальных драм как Мартин Скорцезе, Брайан Де Пальма, Френсис Форд Коппола просто не мог пройти даром. И талантливый актер сделал не менее изящный, подобно десятку своих ролей, режиссерский дебют картиной «Бронкская история».\n\nСыграв также одну из второстепенных ролей, Роберт Де Ниро подарил необычный ему образ правильного и положительного героя, который из всех сил пытается вырастить в своем сыне мужество и справедливость. Ровно, как и герой Чазза Пальминтери, «крестного отца» итальянского квартала, которого, собственно больше боятся, чем любят. \n\nНо и герой Чазза Пальминтери по-своему прав, взяв на воспитание сына Лоренцо, он не утаивает сложность своего места под солнцем, и учит молодого парня в двух направлениях — школы и улицы одновременно.',
 'Название: Однажды в Америке. Отзыв: Естественно вы из всего вышеперечисленного не поймете, почему этот фильм настолько

Попробуем какие-нибудь другие вопросы. На вопрос о наиболее оскароносном фильме получили ответ Титаник. Модель ответила так из-за того, что из всех фильмов в _контексте_ у него больше всего оскаров.

In [33]:
query = 'Какой фильм выиграл больше всего оскаров?'
answer, context = predict(query)

print(answer)

<ipython-input-22-a1fa2b3a3849>:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


Извините, но ваш запрос не содержит информации о том, каким фильмом выиграл больше всего Оскаров. Вы могли бы предоставить дополнительную информацию или изменить свой вопрос?


In [34]:
context.split(' ; ')

['Название: Амадей. Отзыв: Шикарные декорации, потрясающая музыка, отличная актерская игра, прекрасный сценарий оставляют просто великолепные ощущения после просмотра данного киношедевра. Да и 8 «Оскаров» и 3 номинации говорят сами за себя. \n\nОдин из наиболее успешных фильмов в истории Киноакадемии. Браво Милош Форман, браво актеры, браво Моцарт!',
 'Название: Король говорит!. Отзыв: После абсолютно немотивированных главных победителей «Оскара» последних двух лет — «Миллионера из трущоб» и «Повелителя бури» — объявления победителя 2011 года я ждал с изрядной долей опасения. К счастью, опасения оказались напрасными: посмотрев позднее фильм «Король говорит!» я, по крайней мере, могу понять логику рассуждений американских киноакадемиков и согласиться что лента об английском монархе — фильм достойный и стал лучшим по праву. Хотя, признаюсь, я больше бы обрадовался триумфу умного, психологически тонкого «Начала» Кристофера Нолана или простой, но вместе с тем гениальной в своей простоте «Ж

Благодаря промпту модель не отвечает на вопрос, если в контексте нет нужной информации!

In [35]:
query = 'Сколько лет Тому Холланду?'
answer, context = predict(query)

<ipython-input-22-a1fa2b3a3849>:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


In [36]:
answer

'Увы, мне не удалось найти информацию о том, сколько лет Тому Холланду. Возможно, вам стоит обратиться к другим источникам или сайтам, которые специализируются на кинематографе.'

In [37]:
context.split(' ; ')

['Название: Рапунцель Запутанная история. Отзыв: Сам мультфильм находился в разработке ни много ни мало шесть лет, и своему выходу на экран он обязан двум очень хорошим и ответственным людям — Натану Грено и Байрону Ховарду. В этой области творцы мультфильмов уже не новички — Ховард является постановщиком анимационной комедии про пса по имени Вольт, а Натан Грено с четырнадцатилетним трудовым стажем работы на студии принимал участие в создании таких прекрасных творений, как «Мулан» и «Братец медвежонок». Все эти творения номинировались на самые престижные премии, и я уверен, что юбилейная работа Walt Disney Pictures не станет исключением.',
 'Название: Шерлок Холмс Игра теней. Отзыв: - Сколько раз вы собираетесь убивать моего пса Холмс ? (с)',
 'Название: Начало. Отзыв: Эллен Пейдж. Актриса высшего класса. Насколько мне кажется, она не имеет возраста. Ей можно дать 16, а можно и 25… Это редкое качество. Ее героиня помогает зрителям — задает вопросы, объясняет что происходит и как.\n\nД

## Multi-Query

Попробуем расширить контекст, добавив перефразированные вопросы.

<img src="https://i.ibb.co/H4qV7YH/multi-query.png" alt="drawing" width="550"/>

Для перефразирования будем использовать ту же модель, которой генерируем текст, но с новым промптом.

In [38]:
import re

def rephrase_query(query, n=3):
    prompt = f"""
Твоя задача написать {n} разных вариаций вопроса пользователя для того,
чтобы по ним получить релевантные документы из векторной базы данных.
Ты должен переформулировать вопрос с разных точек зрения.
Это поможет избавить пользователя от недостатков поиска похожих документов на основе расстояния.
Вопрос пользователя сфокусирован на теме кино.
Напиши ТОЛЬКО вариации вопроса и больше ничего, разделяя их символом новой строки \\n.
НЕ пиши ответ на сам вопрос.
-----------------
{query}

"""
    messages = [
        {"role": "user", "content": prompt},
    ]
    output = generation_pipeline(messages, max_new_tokens=512, do_sample=True, temperature=0.9, top_p=0.7)
    queries = output[0]['generated_text'][1]['content']

    return re.split(r'\n+', queries)

def predict(query):
    queries = rephrase_query(query, n=3)

    all_chunks = []
    for rephrased_query in queries:
        selected_chunks = semantic_search(client, rephrased_query, limit=5)
        all_chunks.extend(selected_chunks)

    context = [f"Название: {chunk['movie_name']}. Отзыв: {chunk['text']}" for chunk in all_chunks]

    # оставляем только уникальные тексты
    context = ' ; '.join(np.unique(context))

    answer = llm_answer(query, context)

    return answer, context, queries

In [39]:
query = 'Посоветуй легкую комедию'
answer, context, queries = predict(query)

<ipython-input-22-a1fa2b3a3849>:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


In [40]:
print(answer)

Вот несколько легких комедийных фильмов, которые могут быть полезно просмотреть:

1. "Карнавальный бал" (The Great Gatsby) - 9/10
2. "Мой сосед Тоторо" - 10/10
3. "Остров проклятых" - 8/10
4. "Звездные войны: Эпизод III - Месть Ситхов" - 7/10
5. "Моя школа" - 9/10
6. "Престиж" - 10/10
7. "Гладиатор" - 9/10
8. "Мой сосед Тоторо" - 10/10
9. "Леон" - 10/10
10. "Остров проклятых" - 8/10
11. "Шоу Трумана" - 9/10

Эти фильмы имеют высокую оценку у зрителей и предлагают приятные моменты и положительные эмоции. Пожалуйста, учтите, что оценка каждого фильма может отличаться в зависимости от индивидуальных предпочтений и личной точки зрения.


Качество Multi-Query во много зависит от способности LLM перефразировать текст. Если она справляется плохо, то появятся нерелевантные запросы. Это можно исправить, добавив фильтрацию по соответствию входному запросу.

In [41]:
queries

['Кино, подобное "Звездным войнам"',
 'Кино, которое стоит посмотреть по совету друга',
 'Кино, которое можно посмотреть с детьми']

In [42]:
context.split(' ; ')

['Название: Гладиатор. Отзыв: На таких фильмах нужно воспитывать поколения!',
 'Название: Звездные войны Эпизод 3 - Месть Ситхов. Отзыв: Посмотрел я в общем Звездные Войны, как вы и рекомендовали — 4,5,6,1,2,3.',
 'Название: Книга мастеров. Отзыв: А этому фильму 4 из 10',
 'Название: Крамер против Крамера. Отзыв: Душевное кино, к просмотру которого можно возвращаться снова и снова!',
 'Название: Крупная рыба. Отзыв: Посмотрите этот фильм, станьте на два часа детьми и разрешите рассказать вам прекрасную сказку.\n\n10 из 10',
 'Название: Леон. Отзыв: Фильм стоит того, что бы его посмотреть, советую это сделать.\n\n10 из 10',
 'Название: Мой сосед Тоторо. Отзыв: Фильм, который позволяет взрослому человеку вновь почувствовать себя ребёнком. Великолепная вещь! Смотреть всем!\n\n10 из 10',
 'Название: Мой сосед Тоторо. Отзыв: своим детям, которые по праву можно отнести к классики мировой киноиндустрии. Наравне с шедеврами Диснея и СССР.',
 'Название: Начало. Отзыв: К просмотру рекомендуется'

## Фильтры

Для некоторых видов запросов можно добавить фильтры, чтобы в контекст не могли попадать документы, которые однозначно не подходят. Например, мы хотим узнать что-то про фильмы с привязкой к году. Если такой информации нет в самом отзыве, то при семантическом поиске мы не сможем ее использовать.

In [43]:
def predict(query):
    selected_chunks = semantic_search(client, query, limit=10)
    context = ' ; '.join([
        f"Название: {chunk['movie_name']}. Отзыв: {chunk['text']}"
        for chunk in selected_chunks])

    return llm_answer(query, context), context

In [44]:
query = 'Составь список пяти лучших мелодрам 1980-х годов'
answer, context = predict(query)

<ipython-input-22-a1fa2b3a3849>:6: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(


In [45]:
print(answer)

На основе представленных отзывов и их анализ, можно составить список пяти лучших мелодрам 1980-х годов:

1. "Привидение" (1984) - Пожалуй, лучшая мелодрама рубежа 80-х - 90-х, даже несмотря на то, что в это время снято поразительное количество классных мелодрам. Фильм одновременно легкий для восприятия и смысловой. Да-да, многоуважаемые эстеты и снобы - смысловой!

2. "Крамер против Крамера" (1980) - достались не просто так. Как собственно, заслуженно был признан лучшим весь фильм, в 1980 году.

3. "Клуб 'Завтрак'" (1987) - Очень хороший, добрый, веселый фильм. Мне почему то больше нравятся молодежные комедии 80-х. Интересные сюжеты, обалденная музыка, хорошие актеры.

4. "Дневник памяти" (1984) - Отличная игра актеров, хороший сценарий, грамотная режиссура, и вечная тема любви. Тема, которая интересна всегда и везде. Интерес к ней не иссякнет.

5. "Звездный десант 3 Мародер" (1986) - Как любитель космической фантастики + триллера ставлю 8 из 10.


В этом примере модель сгенерировала не только не мелодрамы, но и фильмы неверных лет. Заметьте, что в годах присутствуют галлюцинации.

In [46]:
def filtered_semantic_search(client, query, filter_years, limit=10):
    query_vector = embedding_model.encode(
        query, normalize_embeddings=True, device=device
    ).tolist()

    begin, end = filter_years
    hits = client.search(
        collection_name="kinopoisk_e5",
        query_vector=query_vector,
        limit=limit,
        query_filter=models.Filter(
            must=[models.FieldCondition(key="year", range=models.Range(gte=begin, lte=end))]
        ),
    )
    relevant_chunks = [hit.payload for hit in hits]

    return relevant_chunks

def predict(query, filter_years=None):
    if filter_years is not None:
        selected_chunks = filtered_semantic_search(client, query, filter_years=filter_years, limit=10)
    else:
        selected_chunks = semantic_search(client, query, limit=10)

    context = ' ; '.join([f"Название: {chunk['movie_name']}. Отзыв: {chunk['text']}" for chunk in selected_chunks])

    return llm_answer(query, context), context

In [49]:
answer, context = predict(query, filter_years=(1980, 1989))

<ipython-input-46-a491a8035bc1>:7: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  hits = client.search(
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [50]:
print(answer)

1. "Общество мертвых поэтов"
2. "Клуб "Завтрак""
3. "Лицо со шрамом"
4. "Человек дождя"
5. "Назад в будущее"


Заметьте, что хоть все фильмы и получились нужного года, почти все из них не мелодрамы.

### Резюме

1. Retrieval‑Augmented Generation позволяет улучшить генерацию модели, добавляя в ее контекст полезную информацию.
1. Мы построили RAG для ответов на вопросы по фильмам, который очень неплохо справляется со своей задачей.
1. У всех методов есть свои недостатки и для RAG нет серебряной пули. Для успешной реализации очень важно понимать особенности всех методов и выбирать их в зависимости от данных и задачи.